# BT2 — So sánh ACF và AMDF với ngưỡng Gaussian Mixture Model (GMM)

Notebook này chạy **cả ACF và AMDF** trên cùng các file WAV/LAB, cùng frame shift 10 ms và các frame length 20/25/30 ms. Phần tìm ngưỡng dùng **GMM hai thành phần** thay cho Gaussian intersection một lớp, histogram M1/M2 và giao điểm histogram theo nhãn.

- ACF: score là highest local peak; `score ≥ T` là voiced.
- AMDF: score là lowest local dip; `score < T` là voiced.
- GMM fit trên score của các frame training có nhãn `v` hoặc `uv`, **không dùng nhãn khi fit**. Hai thành phần được sắp theo mean; điểm giao của hai mật độ có trọng số trong khoảng giữa hai mean là ngưỡng `T`.
- Nhãn training chỉ dùng để tính Accuracy V/UV và Balanced accuracy V/UV, rồi chọn frame length. `sil` được loại khỏi bước fit/chọn tham số.
- Trên bốn file test, cả hai thuật toán xuất cùng bảng F0, hai accuracy riêng (`v/uv` và `v/uv/sil`), cùng bảng tổng hợp MAE/RMSE/MAPE và Macro F1.

**Lưu ý:** GMM hai thành phần trên score gộp là mô hình không giám sát. Thành phần có mean cao được gán voiced cho ACF; thành phần có mean thấp được gán voiced cho AMDF. Cùng một quy trình ngưỡng được dùng cho cả hai thuật toán. [GaussianMixture documentation](https://scikit-learn.org/stable/modules/generated/sklearn.mixture.GaussianMixture.html).

## 1. Thư viện và cấu hình

Chạy trong working folder local chứa `TinHieuHuanLuyen` và `TinHieuKiemThu`. Có thể đặt `BT2_PROJECT_DIR` nếu chạy từ thư mục khác. Output đã lưu là mốc tham khảo; **Run All** để tạo kết quả local.

In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.signal import correlate
from sklearn.mixture import GaussianMixture

FRAME_CANDIDATES_MS = (20, 25, 30)
HOP_MS = 10
F0_MIN = 70.0
F0_MAX = 400.0
EPS = 1e-12
NEAR_ZERO_MEAN_ABS = 1e-8

PROJECT_DIR = Path(os.environ.get('BT2_PROJECT_DIR', Path.cwd())).expanduser().resolve()
TRAIN_DIR = PROJECT_DIR / 'TinHieuHuanLuyen'
TEST_DIR = PROJECT_DIR / 'TinHieuKiemThu'
if not TRAIN_DIR.is_dir() or not TEST_DIR.is_dir():
    raise FileNotFoundError(f'Không tìm thấy TinHieuHuanLuyen/TinHieuKiemThu trong {PROJECT_DIR}')

print('PROJECT_DIR:', PROJECT_DIR)
print('Frame lengths (ms):', FRAME_CANDIDATES_MS, '| frame shift (ms):', HOP_MS)

## 2. Đọc WAV/LAB và chia khung

Hai thuật toán dùng **cùng** hàm đọc file, chuẩn hóa biên độ, chia khung và lấy nhãn theo tâm khung. Các frame sát biên segment có thể chứa mẫu từ hai nhãn; quy tắc gán nhãn vẫn giống nhau cho ACF và AMDF.

In [ ]:
def load_wav(path):
    fs, signal = wavfile.read(path)
    if signal.ndim > 1:
        signal = signal.mean(axis=1)
    if np.issubdtype(signal.dtype, np.integer):
        info = np.iinfo(signal.dtype)
        signal = signal.astype(np.float64) / max(abs(info.min), info.max)
    else:
        signal = signal.astype(np.float64)
    return int(fs), signal


def read_lab(path):
    segments, f0_mean, f0_std = [], None, None
    with open(path, 'r', encoding='utf-8') as handle:
        for raw_line in handle:
            parts = raw_line.strip().split()
            if not parts:
                continue
            if parts[0] == 'F0mean':
                f0_mean = float(parts[1])
            elif parts[0] == 'F0std':
                f0_std = float(parts[1])
            else:
                segments.append({
                    'start': float(parts[0]),
                    'end': float(parts[1]),
                    'label': parts[2].lower(),
                })
    return segments, f0_mean, f0_std


def pair_wav_lab(folder):
    pairs = []
    for wav_path in sorted(folder.glob('*.wav')):
        lab_path = wav_path.with_suffix('.lab')
        if not lab_path.exists():
            raise FileNotFoundError(f'Thiếu LAB cho {wav_path.name}')
        pairs.append((wav_path, lab_path))
    return pairs


def framing(signal, fs, frame_ms, hop_ms=HOP_MS):
    frame_len = int(round(fs * frame_ms / 1000.0))
    hop_len = int(round(fs * hop_ms / 1000.0))
    if len(signal) < frame_len:
        return np.empty((0, frame_len)), np.empty(0)
    frames = np.lib.stride_tricks.sliding_window_view(signal, frame_len)[::hop_len]
    starts = np.arange(len(frames)) * hop_len
    center_times = (starts + frame_len / 2.0) / fs
    return frames, center_times


def label_at_time(time_sec, segments):
    for segment in segments:
        if segment['start'] <= time_sec < segment['end']:
            return segment['label']
    return None


train_pairs = pair_wav_lab(TRAIN_DIR)
test_pairs = pair_wav_lab(TEST_DIR)
assert train_pairs, 'Tập huấn luyện không có cặp WAV/LAB.'
assert len(test_pairs) == 4, f'Đề bài cần 4 file test; tìm thấy {len(test_pairs)}.'
print('Training WAV:', [wav.name for wav, _ in train_pairs])
print('Testing WAV :', [wav.name for wav, _ in test_pairs])

## 3. Trích score và lag ACF/AMDF

Các hàm score/lag được giữ từ hai notebook đã đồng bộ. ACF lấy highest local peak của normalized ACF; AMDF lấy lowest local dip của normalized AMDF. Cả hai giới hạn F0 trong 70–400 Hz và nội suy parabol quanh lag tốt nhất.

In [ ]:
# -------- ACF: normalized score, highest local peak và F0 --------
def normalized_acf(frame):
    x = np.asarray(frame, dtype=np.float64)
    x = x - np.mean(x)
    n = len(x)

    if n == 0:
        return np.array([])

    # Numerator ACF cho các lag >= 0.
    # FFT correlation giúp quét toàn bộ lag nhanh hơn loop Python.
    numerator = correlate(x, x, mode='full', method='fft')[n - 1:]

    # Vector hóa mẫu số theo đúng vùng overlap của từng lag.
    sq = x * x
    csum = np.concatenate(([0.0], np.cumsum(sq)))
    lags = np.arange(n)

    energy_left = csum[n - lags]       # sum x[0 : n-k]^2
    energy_right = csum[n] - csum[lags]  # sum x[k : n]^2
    denominator = np.sqrt(energy_left * energy_right)

    acf = np.divide(
        numerator,
        denominator,
        out=np.zeros_like(numerator, dtype=np.float64),
        where=denominator > EPS
    )

    return acf

def detect_pitch_acf(frame, fs, f0_min=F0_MIN, f0_max=F0_MAX):
    acf = normalized_acf(frame)

    lag_min = max(1, int(np.ceil(fs / f0_max)))
    lag_max = min(len(frame) - 2, int(np.floor(fs / f0_min)))

    if lag_min > lag_max:
        return 0.0, np.nan, acf

    candidate_lags = np.arange(lag_min, lag_max + 1)

    # Local peak: >= hai hàng xóm.
    local_mask = (
        (acf[candidate_lags] >= acf[candidate_lags - 1]) &
        (acf[candidate_lags] >= acf[candidate_lags + 1])
    )
    local_lags = candidate_lags[local_mask]

    if len(local_lags) > 0:
        best_lag_int = int(local_lags[np.argmax(acf[local_lags])])
    else:
        # Trường hợp hiếm: không có local peak -> lấy max trong search range.
        best_lag_int = int(
            lag_min + np.argmax(acf[lag_min:lag_max + 1])
        )

    periodicity_score = float(acf[best_lag_int])

    # Parabolic interpolation quanh peak để tinh chỉnh lag.
    best_lag = float(best_lag_int)
    y_m1 = acf[best_lag_int - 1]
    y_0 = acf[best_lag_int]
    y_p1 = acf[best_lag_int + 1]
    denom = y_m1 - 2.0 * y_0 + y_p1

    if abs(denom) > EPS:
        delta = 0.5 * (y_m1 - y_p1) / denom
        if abs(delta) <= 1.0:
            best_lag += float(delta)

    return periodicity_score, best_lag, acf


def lag_to_f0(best_lag, fs):
    if not np.isfinite(best_lag) or best_lag <= 0:
        return np.nan
    return fs / best_lag

# -------- AMDF: normalized score, lowest local dip và F0 --------
def lag_search_range(fs, frame_len, f0_min=F0_MIN, f0_max=F0_MAX):
    """Tính miền lag hợp lệ từ dải F0."""
    lag_min = max(1, int(np.floor(fs / f0_max)))
    lag_max = min(frame_len - 1, int(np.ceil(fs / f0_min)))
    if lag_min > lag_max:
        raise ValueError('Frame quá ngắn so với miền lag cần tìm.')
    return lag_min, lag_max


def normalized_amdf(frame, fs, f0_min=F0_MIN, f0_max=F0_MAX):
    """
    Tính normalized AMDF trong miền lag hợp lệ.

    Vector hóa bằng NumPy theo cả trục lag và sample; chỉ tạo ma trận nhỏ
    tương ứng các lag 70–400 Hz, không tính các lag không cần thiết.
    """
    x = np.asarray(frame, dtype=np.float64)
    x = x - np.mean(x)  # loại DC nhỏ nếu có

    lag_min, lag_max = lag_search_range(fs, len(x), f0_min, f0_max)
    lags = np.arange(lag_min, lag_max + 1, dtype=int)

    # Frame gần như toàn số 0: AMDF chuẩn hóa bị suy biến 0/0.
    # Gán giá trị 1.0 để biểu diễn "không có bằng chứng tuần hoàn hữu ích".
    if np.mean(np.abs(x)) < NEAR_ZERO_MEAN_ABS:
        return lags, np.ones_like(lags, dtype=np.float64)

    max_overlap = len(x) - lag_min
    n = np.arange(max_overlap)[None, :]
    lag_matrix = lags[:, None]

    valid = n < (len(x) - lag_matrix)
    idx2 = np.minimum(n + lag_matrix, len(x) - 1)

    x1 = np.broadcast_to(x[:max_overlap][None, :], idx2.shape)
    x2 = x[idx2]

    count = valid.sum(axis=1)
    diff_mean = (np.abs(x1 - x2) * valid).sum(axis=1) / count
    x1_mean_abs = (np.abs(x1) * valid).sum(axis=1) / count
    x2_mean_abs = (np.abs(x2) * valid).sum(axis=1) / count

    namdf = diff_mean / (x1_mean_abs + x2_mean_abs + EPS)
    return lags, namdf


def deepest_local_dip(frame, fs, f0_min=F0_MIN, f0_max=F0_MAX):
    """
    Tìm lowest local dip của normalized AMDF.

    Returns
    -------
    dip_value : float
        Giá trị đáy AMDF dùng làm periodicity score.
    lag : int
        Lag nguyên tại đáy.
    refined_lag : float
        Lag tinh chỉnh bằng nội suy parabol để ước lượng F0 mượt hơn.
    lags, namdf : ndarray
        Dùng cho vẽ minh họa.
    """
    lags, namdf = normalized_amdf(frame, fs, f0_min, f0_max)

    if len(namdf) >= 3:
        local_mask = (namdf[1:-1] <= namdf[:-2]) & (namdf[1:-1] <= namdf[2:])
        local_idx = np.where(local_mask)[0] + 1
    else:
        local_idx = np.array([], dtype=int)

    # Nếu không có cực tiểu cục bộ nội bộ, fallback về minimum toàn cục.
    if len(local_idx) == 0:
        best_idx = int(np.argmin(namdf))
    else:
        best_idx = int(local_idx[np.argmin(namdf[local_idx])])

    dip_value = float(namdf[best_idx])
    lag = int(lags[best_idx])
    refined_lag = float(lag)

    # Nội suy parabol quanh đáy để giảm sai số lượng tử theo sample.
    if 0 < best_idx < len(namdf) - 1:
        y1, y2, y3 = namdf[best_idx - 1: best_idx + 2]
        denominator = y1 - 2.0 * y2 + y3
        if abs(denominator) > EPS:
            delta = 0.5 * (y1 - y3) / denominator
            if abs(delta) <= 1.0:
                refined_lag += float(delta)

    return dip_value, lag, refined_lag, lags, namdf


def f0_from_lag(fs, lag):
    return fs / lag if lag > 0 else np.nan

## 4. Thu thập score training

Mỗi tổ hợp thuật toán × frame length dùng đúng cùng bốn file training và cùng quy tắc lấy nhãn. Chỉ score của `v` và `uv` đi vào GMM.

In [ ]:
def score_frame(frame, fs, algorithm):
    if algorithm == 'ACF':
        score, lag, curve = detect_pitch_acf(frame, fs)
        return score, lag, np.arange(len(curve)), curve
    if algorithm == 'AMDF':
        score, _, lag, lags, curve = deepest_local_dip(frame, fs)
        return score, lag, lags, curve
    raise ValueError(f'Unknown algorithm: {algorithm}')


def collect_training_scores(algorithm, frame_ms):
    rows = []
    for wav_path, lab_path in train_pairs:
        fs, signal = load_wav(wav_path)
        segments, _, _ = read_lab(lab_path)
        frames, times = framing(signal, fs, frame_ms)
        for frame, time_sec in zip(frames, times):
            label = label_at_time(time_sec, segments)
            if label not in {'v', 'uv'}:
                continue
            score, lag, _, _ = score_frame(frame, fs, algorithm)
            rows.append({
                'File': wav_path.name, 'Center (s)': time_sec,
                'Label': label, 'Score': score,
                'Best lag (samples)': lag,
            })
    return pd.DataFrame(rows)


training_scores = {
    (algorithm, frame_ms): collect_training_scores(algorithm, frame_ms)
    for algorithm in ('ACF', 'AMDF')
    for frame_ms in FRAME_CANDIDATES_MS
}

## 5. Fit GMM hai thành phần và chọn ngưỡng

Với hai thành phần được sắp theo mean μ₁ < μ₂, notebook tìm `T` giữa hai mean sao cho:

\[
w_1\mathcal{N}(T;\mu_1,\sigma_1^2)=w_2\mathcal{N}(T;\mu_2,\sigma_2^2).
\]

Nếu phép lấy mẫu số trên khoảng giữa hai mean không cho giao điểm đổi dấu, `T` là vị trí trong khoảng có sai khác mật độ nhỏ nhất. GMM fit trên **score gộp không nhãn**; `.lab` chỉ dùng để đo Accuracy/Balanced accuracy và chọn frame length trên training.

In [ ]:
def normal_pdf(x, mean, std):
    std = max(float(std), EPS)
    return np.exp(-0.5 * ((x - mean) / std) ** 2) / (std * np.sqrt(2.0 * np.pi))


def fit_gmm_threshold(scores):
    scores = np.asarray(scores, dtype=float)
    scores = scores[np.isfinite(scores)]
    if len(scores) < 4 or np.std(scores) < 1e-10:
        raise ValueError('Score training quá ít hoặc không biến thiên để fit GMM hai thành phần.')
    model = GaussianMixture(
        n_components=2, covariance_type='spherical',
        reg_covar=1e-6, n_init=10, random_state=42,
    ).fit(scores.reshape(-1, 1))
    order = np.argsort(model.means_.ravel())
    means = model.means_.ravel()[order]
    stds = np.sqrt(model.covariances_[order])
    weights = model.weights_[order]

    grid = np.linspace(means[0], means[1], 10001)
    low_density = weights[0] * normal_pdf(grid, means[0], stds[0])
    high_density = weights[1] * normal_pdf(grid, means[1], stds[1])
    difference = low_density - high_density
    crossing_indices = np.where(difference[:-1] * difference[1:] <= 0)[0]
    if len(crossing_indices):
        midpoint = 0.5 * (means[0] + means[1])
        index = min(crossing_indices, key=lambda i: abs(grid[i] - midpoint))
        left, right = difference[index], difference[index + 1]
        fraction = -left / (right - left) if right != left else 0.0
        threshold = float(grid[index] + fraction * (grid[index + 1] - grid[index]))
    else:
        threshold = float(grid[np.argmin(np.abs(difference))])
    return {'model': model, 'means': means, 'stds': stds,
            'weights': weights, 'threshold': threshold}


def classification_metrics(labels, scores, threshold, algorithm):
    labels = np.asarray(labels)
    scores = np.asarray(scores, dtype=float)
    true_voiced = labels == 'v'
    pred_voiced = scores >= threshold if algorithm == 'ACF' else scores < threshold
    tp = int(np.sum(true_voiced & pred_voiced))
    tn = int(np.sum(~true_voiced & ~pred_voiced))
    fp = int(np.sum(~true_voiced & pred_voiced))
    fn = int(np.sum(true_voiced & ~pred_voiced))
    accuracy = (tp + tn) / len(labels) if len(labels) else np.nan
    recall_v = tp / (tp + fn) if (tp + fn) else 0.0
    recall_uv = tn / (tn + fp) if (tn + fp) else 0.0
    return {'accuracy': accuracy, 'balanced_accuracy': 0.5 * (recall_v + recall_uv),
            'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn}


gmm_models = {}
training_rows = []
for algorithm in ('ACF', 'AMDF'):
    for frame_ms in FRAME_CANDIDATES_MS:
        data = training_scores[(algorithm, frame_ms)]
        v = data.loc[data['Label'] == 'v', 'Score'].to_numpy()
        uv = data.loc[data['Label'] == 'uv', 'Score'].to_numpy()
        if not len(v) or not len(uv):
            raise ValueError(f'Thiếu frame V hoặc UV trong training: {algorithm}, {frame_ms} ms')
        gmm = fit_gmm_threshold(data['Score'].to_numpy())
        metrics = classification_metrics(data['Label'], data['Score'],
                                         gmm['threshold'], algorithm)
        gmm_models[(algorithm, frame_ms)] = {'gmm': gmm, 'metrics': metrics}
        training_rows.append({
            'Algorithm': algorithm, 'Frame (ms)': frame_ms,
            'N V': len(v), 'N UV': len(uv),
            'meanV': float(v.mean()), 'stdV': float(v.std()),
            'meanU': float(uv.mean()), 'stdU': float(uv.std()),
            'GMM μlow': gmm['means'][0], 'GMM μhigh': gmm['means'][1],
            'Weight low': gmm['weights'][0], 'Weight high': gmm['weights'][1],
            'Threshold': gmm['threshold'],
            'Accuracy V/UV': metrics['accuracy'],
            'Balanced accuracy V/UV': metrics['balanced_accuracy'],
        })

training_summary_df = pd.DataFrame(training_rows)
training_display = training_summary_df.copy()
for column in ['meanV', 'stdV', 'meanU', 'stdU', 'GMM μlow', 'GMM μhigh',
               'Weight low', 'Weight high', 'Threshold']:
    training_display[column] = training_display[column].map(lambda value: f'{value:.4f}')
for column in ['Accuracy V/UV', 'Balanced accuracy V/UV']:
    training_display[column] = training_display[column].map(lambda value: f'{value:.2%}')
print('GMM threshold on training V/UV frames:')
display(training_display)

selected_models = {}
selection_rows = []
for algorithm in ('ACF', 'AMDF'):
    candidates = [row for row in training_rows if row['Algorithm'] == algorithm]
    best = max(candidates, key=lambda row: (
        row['Balanced accuracy V/UV'], row['Accuracy V/UV']
    ))
    selected_models[algorithm] = gmm_models[(algorithm, best['Frame (ms)'])]
    selected_models[algorithm]['frame_ms'] = best['Frame (ms)']
    selection_rows.append({
        'Algorithm': algorithm, 'Frame (ms)': best['Frame (ms)'],
        'Method': 'GMM (2 components)', 'Threshold': best['Threshold'],
        'Accuracy V/UV': best['Accuracy V/UV'],
        'Balanced accuracy V/UV': best['Balanced accuracy V/UV'],
    })
selection_df = pd.DataFrame(selection_rows)
selection_display = selection_df.copy()
selection_display['Threshold'] = selection_display['Threshold'].map(lambda value: f'{value:.4f}')
for column in ['Accuracy V/UV', 'Balanced accuracy V/UV']:
    selection_display[column] = selection_display[column].map(lambda value: f'{value:.2%}')
print('Final configuration chosen on training:')
display(selection_display)

## 6. Đồ thị GMM cho ACF và AMDF

Mỗi thuật toán có một figure gồm ba panel 20/25/30 ms. Histogram `v/uv` chỉ để kiểm tra trực quan; GMM được fit trên score gộp. Đường đứng là ngưỡng từ giao điểm hai thành phần có trọng số.

In [ ]:
for algorithm in ('ACF', 'AMDF'):
    fig, axes = plt.subplots(1, len(FRAME_CANDIDATES_MS),
                             figsize=(5.5 * len(FRAME_CANDIDATES_MS), 4.8))
    for ax, frame_ms in zip(axes, FRAME_CANDIDATES_MS):
        data = training_scores[(algorithm, frame_ms)]
        v = data.loc[data['Label'] == 'v', 'Score'].to_numpy()
        uv = data.loc[data['Label'] == 'uv', 'Score'].to_numpy()
        info = gmm_models[(algorithm, frame_ms)]
        gmm = info['gmm']
        x = np.linspace(data['Score'].min(), data['Score'].max(), 500)
        low = gmm['weights'][0] * normal_pdf(x, gmm['means'][0], gmm['stds'][0])
        high = gmm['weights'][1] * normal_pdf(x, gmm['means'][1], gmm['stds'][1])
        ax.hist(v, bins=30, density=True, alpha=0.25, label='GT V')
        ax.hist(uv, bins=30, density=True, alpha=0.25, label='GT UV')
        ax.plot(x, low, linewidth=2, label='Weighted component low')
        ax.plot(x, high, linewidth=2, label='Weighted component high')
        ax.plot(x, low + high, linestyle=':', linewidth=2, label='GMM mixture')
        ax.axvline(gmm['threshold'], linestyle='--', linewidth=2,
                   label=f"Threshold={gmm['threshold']:.4f}")
        ax.set_title(f"{algorithm} · {frame_ms} ms · BalAcc {info['metrics']['balanced_accuracy']:.2%}")
        ax.set_xlabel('Highest local ACF peak' if algorithm == 'ACF' else 'Lowest local AMDF dip')
        ax.set_ylabel('Density')
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()

## 7. Demo cùng file và thời điểm tâm

Ví dụ `v` và `uv` được chọn theo thứ tự file/segment LAB (segment đầu tiên dài ít nhất 50 ms), độc lập với score. Vì vậy ACF và AMDF minh họa **cùng hai thời điểm**, mỗi thời điểm được khảo sát ở 20/25/30 ms.

In [ ]:
def find_demo_segment(target_label):
    for wav_path, lab_path in train_pairs:
        segments, _, _ = read_lab(lab_path)
        for segment in segments:
            if segment['label'] == target_label and segment['end'] - segment['start'] >= 0.05:
                return {'wav_path': wav_path, 'time': 0.5 * (segment['start'] + segment['end']),
                        'label': target_label}
    raise RuntimeError(f'Không tìm thấy segment {target_label} đủ dài.')


def extract_centered_frame(signal, fs, center_time, frame_ms):
    frame_len = int(round(fs * frame_ms / 1000.0))
    center = int(round(center_time * fs))
    start = max(0, min(center - frame_len // 2, len(signal) - frame_len))
    return signal[start:start + frame_len]


demo_rows = []
for target_label in ('v', 'uv'):
    candidate = find_demo_segment(target_label)
    fs, signal = load_wav(candidate['wav_path'])
    for algorithm in ('ACF', 'AMDF'):
        fig, axes = plt.subplots(1, len(FRAME_CANDIDATES_MS),
                                 figsize=(5.3 * len(FRAME_CANDIDATES_MS), 4.5))
        for ax, frame_ms in zip(axes, FRAME_CANDIDATES_MS):
            frame = extract_centered_frame(signal, fs, candidate['time'], frame_ms)
            score, lag, lags, curve = score_frame(frame, fs, algorithm)
            threshold = gmm_models[(algorithm, frame_ms)]['gmm']['threshold']
            pred_voiced = score >= threshold if algorithm == 'ACF' else score < threshold
            f0 = fs / lag if pred_voiced and np.isfinite(lag) and lag > 0 else np.nan
            ax.plot(lags / fs * 1000.0, curve, label=f'{algorithm} curve')
            ax.scatter([lag / fs * 1000.0], [score], s=60, label=f'Score={score:.3f}')
            ax.axhline(threshold, linestyle='--', label=f'Threshold={threshold:.3f}')
            ax.set_title(f'{algorithm} · GT {target_label} · {frame_ms} ms')
            ax.set_xlabel('Lag (ms)')
            ax.set_ylabel('Normalized score')
            ax.grid(alpha=0.25)
            ax.legend(fontsize=8)
            demo_rows.append({
                'Algorithm': algorithm, 'Example': 'Voiced' if target_label == 'v' else 'Unvoiced',
                'File': candidate['wav_path'].name, 'Center (s)': candidate['time'],
                'GT': target_label, 'Frame (ms)': frame_ms, 'Score': score,
                'Threshold': threshold, 'Predicted': 'v' if pred_voiced else 'uv',
                'Best lag (samples)': lag, 'F0 (Hz)': f0,
            })
        fig.tight_layout()
        plt.show()

demo_df = pd.DataFrame(demo_rows)
print('Same-center V/UV examples across algorithms and frame lengths:')
display(demo_df.round(4))

## 8. Kiểm thử bốn file cho từng thuật toán

Mỗi file tạo một figure waveform + F0 contour cho ACF và một figure tương tự cho AMDF. Các bảng dưới có cùng tên cột cho hai thuật toán; cột `Algorithm` xác định dòng thuộc ACF hay AMDF.

`Accuracy V/UV` chỉ xét `v` và `uv`; `Accuracy V/UV/SIL` xét thêm `sil` dưới nhãn non-voiced. `NumF0` là số khung có F0 ước lượng hữu hạn. Từng file chỉ có một F0mean/F0std trong LAB, nên cột MAE của dòng bằng sai số tuyệt đối của thống kê đó.

In [ ]:
def macro_f1_vu(labels, predicted_voiced):
    labels = np.asarray(labels)
    pred = np.asarray(predicted_voiced, dtype=bool)
    mask = np.isin(labels, ['v', 'uv'])
    truth = labels[mask] == 'v'
    pred = pred[mask]
    tp = int(np.sum(truth & pred))
    tn = int(np.sum(~truth & ~pred))
    fp = int(np.sum(~truth & pred))
    fn = int(np.sum(truth & ~pred))
    f1_v = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0
    f1_uv = 2 * tn / (2 * tn + fp + fn) if (2 * tn + fp + fn) else 0.0
    return 0.5 * (f1_v + f1_uv) if mask.any() else np.nan


def process_test_file(wav_path, lab_path, algorithm, frame_ms, threshold):
    fs, signal = load_wav(wav_path)
    segments, gt_mean, gt_std = read_lab(lab_path)
    frames, times = framing(signal, fs, frame_ms)
    scores = np.empty(len(frames))
    lags = np.empty(len(frames))
    predicted = np.zeros(len(frames), dtype=bool)
    f0 = np.full(len(frames), np.nan)
    labels = np.empty(len(frames), dtype=object)
    for i, (frame, time_sec) in enumerate(zip(frames, times)):
        labels[i] = label_at_time(time_sec, segments)
        score, lag, _, _ = score_frame(frame, fs, algorithm)
        scores[i], lags[i] = score, lag
        predicted[i] = score >= threshold if algorithm == 'ACF' else score < threshold
        if predicted[i] and np.isfinite(lag) and lag > 0:
            f0[i] = fs / lag

    mask_vu = np.isin(labels, ['v', 'uv'])
    mask_all = np.isin(labels, ['v', 'uv', 'sil'])
    true_voiced = labels == 'v'
    correct_vu = int(np.sum(true_voiced[mask_vu] == predicted[mask_vu]))
    correct_all = int(np.sum(true_voiced[mask_all] == predicted[mask_all]))
    n_vu, n_all = int(mask_vu.sum()), int(mask_all.sum())
    finite_f0 = f0[np.isfinite(f0)]
    est_mean = float(np.mean(finite_f0)) if len(finite_f0) else np.nan
    est_std = float(np.std(finite_f0)) if len(finite_f0) else np.nan
    return {
        'Algorithm': algorithm, 'file': wav_path.name,
        'fs': fs, 'signal': signal, 'segments': segments,
        'times': times, 'labels': labels, 'scores': scores,
        'predicted_voiced': predicted, 'f0': f0,
        'F0mean_GT': gt_mean, 'F0mean_est': est_mean,
        'abs_error_mean': abs(est_mean - gt_mean) if gt_mean is not None and np.isfinite(est_mean) else np.nan,
        'F0std_GT': gt_std, 'F0std_est': est_std,
        'abs_error_std': abs(est_std - gt_std) if gt_std is not None and np.isfinite(est_std) else np.nan,
        'NumF0': int(len(finite_f0)),
        'TP': int(np.sum(mask_vu & true_voiced & predicted)),
        'FN': int(np.sum(mask_vu & true_voiced & ~predicted)),
        'FP': int(np.sum(mask_vu & ~true_voiced & predicted)),
        'TN': int(np.sum(mask_vu & ~true_voiced & ~predicted)),
        'correct_vu': correct_vu, 'n_vu': n_vu,
        'accuracy_vu': correct_vu / n_vu if n_vu else np.nan,
        'correct_all': correct_all, 'n_all': n_all,
        'accuracy_all': correct_all / n_all if n_all else np.nan,
        'sil_false_voiced': int(np.sum((labels == 'sil') & predicted)),
        'macro_f1_vu': macro_f1_vu(labels, predicted),
    }


def plot_test_result(result):
    signal, fs = result['signal'], result['fs']
    times, f0 = result['times'], result['f0']
    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
    axes[0].plot(np.arange(len(signal)) / fs, signal, linewidth=0.8)
    first_gt = True
    for segment in result['segments']:
        if segment['label'] == 'v':
            axes[0].axvspan(segment['start'], segment['end'], color='blue', alpha=0.12,
                            label='Ground-truth voiced' if first_gt else None)
            first_gt = False
    pred = result['predicted_voiced']
    in_voiced, first_pred = False, True
    hop_s = HOP_MS / 1000.0
    for i, is_voiced in enumerate(pred):
        if is_voiced and not in_voiced:
            start_t = times[i] - hop_s / 2
            in_voiced = True
        elif not is_voiced and in_voiced:
            axes[0].axvspan(start_t, times[i - 1] + hop_s / 2, color='green', alpha=0.25,
                            label='Predicted voiced' if first_pred else None)
            first_pred, in_voiced = False, False
    if in_voiced:
        axes[0].axvspan(start_t, times[-1] + hop_s / 2, color='green', alpha=0.25,
                        label='Predicted voiced' if first_pred else None)
    axes[0].set_title(f"{result['Algorithm']} · Input waveform – {result['file']}")
    axes[0].set_ylabel('Amplitude')
    axes[0].grid(alpha=0.25)
    axes[0].legend(loc='upper right')

    axes[1].plot(times, f0, '.-', markersize=3, linewidth=1, label='Estimated F0 contour')
    gt_mean, gt_std = result['F0mean_GT'], result['F0std_GT']
    if gt_mean is not None:
        axes[1].axhline(gt_mean, linestyle='--', label=f'LAB F0mean={gt_mean:.1f} Hz')
    if gt_mean is not None and gt_std is not None:
        axes[1].axhspan(gt_mean - gt_std, gt_mean + gt_std, alpha=0.10,
                        label=f'LAB F0mean ± F0std ({gt_std:.1f} Hz)')
    axes[1].set_ylim(0, F0_MAX + 30)
    axes[1].set_xlabel('Thời gian (s)')
    axes[1].set_ylabel('F0 (Hz)')
    axes[1].set_title(
        f"Output – F0mean={result['F0mean_est']:.2f} Hz, "
        f"F0std={result['F0std_est']:.2f} Hz, "
        f"Acc(V/UV)={result['accuracy_vu']:.2%}, "
        f"Acc(V/UV/SIL)={result['accuracy_all']:.2%}"
    )
    axes[1].grid(alpha=0.25)
    axes[1].legend(loc='upper right')
    plt.tight_layout()
    plt.show()


test_rows = []
for algorithm in ('ACF', 'AMDF'):
    chosen = selected_models[algorithm]
    for wav_path, lab_path in test_pairs:
        result = process_test_file(wav_path, lab_path, algorithm,
                                   chosen['frame_ms'], chosen['gmm']['threshold'])
        plot_test_result(result)
        test_rows.append({key: value for key, value in result.items()
                          if key not in {'fs', 'signal', 'segments', 'times', 'labels',
                                         'scores', 'predicted_voiced', 'f0'}})

test_summary_df = pd.DataFrame(test_rows)
f0_report = test_summary_df[[
    'Algorithm', 'file', 'F0mean_GT', 'F0mean_est', 'abs_error_mean',
    'F0std_GT', 'F0std_est', 'abs_error_std', 'NumF0'
]].rename(columns={
    'file': 'TEST WAV',
    'F0mean_GT': 'LAB F0mean', 'F0mean_est': 'Pred F0mean',
    'abs_error_mean': 'F0mean MAE',
    'F0std_GT': 'LAB F0std', 'F0std_est': 'Pred F0std',
    'abs_error_std': 'F0std MAE',
})
print('Per-file F0 metrics (all F0 values in Hz; NumF0 is a frame count):')
display(f0_report.round(4))

classification_report = test_summary_df[[
    'Algorithm', 'file', 'correct_vu', 'n_vu', 'accuracy_vu',
    'correct_all', 'n_all', 'accuracy_all', 'sil_false_voiced'
]].rename(columns={
    'file': 'TEST WAV',
    'correct_vu': 'Correct V/UV', 'n_vu': 'N V/UV',
    'accuracy_vu': 'Accuracy V/UV',
    'correct_all': 'Correct V/UV/SIL', 'n_all': 'N V/UV/SIL',
    'accuracy_all': 'Accuracy V/UV/SIL',
    'sil_false_voiced': 'False voiced on SIL',
})
classification_display = classification_report.copy()
for column in ['Accuracy V/UV', 'Accuracy V/UV/SIL']:
    classification_display[column] = classification_display[column].map(
        lambda value: f'{value:.2%}' if pd.notna(value) else '—'
    )
print('Per-file classification accuracy:')
display(classification_display)

## 9. Tổng hợp và so sánh ACF/AMDF

MAE/RMSE/MAPE của F0mean và F0std được tính từ bốn cặp thống kê LAB/dự đoán theo file; MAPE bỏ GT bằng 0. `Macro F1 (V/UV)` là trung bình F1 của hai lớp trong từng file, rồi trung bình không trọng số qua bốn file. Hai macro accuracy cũng là trung bình không trọng số qua bốn file. Bảng hiển thị đơn sắc.

In [ ]:
def aggregate_f0_metrics(gt, pred):
    gt = np.asarray(gt, dtype=float)
    pred = np.asarray(pred, dtype=float)
    valid = np.isfinite(gt) & np.isfinite(pred)
    error = pred[valid] - gt[valid]
    mae = float(np.mean(np.abs(error))) if len(error) else np.nan
    rmse = float(np.sqrt(np.mean(error ** 2))) if len(error) else np.nan
    percent_valid = valid & (gt != 0)
    mape = (float(np.mean(np.abs((pred[percent_valid] - gt[percent_valid]) / gt[percent_valid])))
            if percent_valid.any() else np.nan)
    return mae, rmse, mape


f0_aggregate_rows = []
accuracy_aggregate_rows = []
for algorithm, group in test_summary_df.groupby('Algorithm', sort=False):
    mean_mae, mean_rmse, mean_mape = aggregate_f0_metrics(
        group['F0mean_GT'], group['F0mean_est'])
    std_mae, std_rmse, std_mape = aggregate_f0_metrics(
        group['F0std_GT'], group['F0std_est'])
    f0_aggregate_rows.append({
        'Algorithm': algorithm, 'files': len(group),
        'F0mean MAE': mean_mae, 'F0mean RMSE': mean_rmse,
        'F0mean MAPE': mean_mape,
        'F0std MAE': std_mae, 'F0std RMSE': std_rmse,
        'F0std MAPE': std_mape,
        'Macro F1 (V/UV)': group['macro_f1_vu'].mean(),
    })
    accuracy_aggregate_rows.append({
        'Algorithm': algorithm, 'files': len(group),
        'Macro accuracy V/UV': group['accuracy_vu'].mean(),
        'Macro accuracy V/UV/SIL': group['accuracy_all'].mean(),
    })

f0_aggregate = pd.DataFrame(f0_aggregate_rows)
f0_aggregate_display = f0_aggregate.copy()
for column in ['F0mean MAE', 'F0mean RMSE', 'F0std MAE', 'F0std RMSE', 'Macro F1 (V/UV)']:
    f0_aggregate_display[column] = f0_aggregate_display[column].map(
        lambda value: f'{value:.4f}' if pd.notna(value) else '—')
for column in ['F0mean MAPE', 'F0std MAPE']:
    f0_aggregate_display[column] = f0_aggregate_display[column].map(
        lambda value: f'{value:.2%}' if pd.notna(value) else '—')
print('Aggregate F0 metrics (MAE/RMSE in Hz; MAPE in %):')
display(f0_aggregate_display)

accuracy_aggregate = pd.DataFrame(accuracy_aggregate_rows)
accuracy_aggregate_display = accuracy_aggregate.copy()
for column in ['Macro accuracy V/UV', 'Macro accuracy V/UV/SIL']:
    accuracy_aggregate_display[column] = accuracy_aggregate_display[column].map(
        lambda value: f'{value:.2%}' if pd.notna(value) else '—')
print('Aggregate classification accuracy:')
display(accuracy_aggregate_display)

In [ ]:
# Chẩn đoán V/UV: số khung và tỷ lệ theo nhãn thật trên tập test.
# Đây là mô tả cấu hình đã chọn từ training; không dùng hình này để chọn ngưỡng.
if 'Algorithm' in test_summary_df.columns:
    confusion_groups = list(test_summary_df.groupby('Algorithm', sort=False))
else:
    confusion_groups = [('V/UV', test_summary_df)]
fig, axes = plt.subplots(1, len(confusion_groups), figsize=(5.2 * len(confusion_groups), 4.4), squeeze=False, layout='constrained')
for ax, (name, group) in zip(axes[0], confusion_groups):
    cm = np.array([[group['TP'].sum(), group['FN'].sum()],
                   [group['FP'].sum(), group['TN'].sum()]], dtype=int)
    assert int(cm.sum()) == int(group['n_vu'].sum())
    assert int(cm[0, 0] + cm[1, 1]) == int(group['correct_vu'].sum())
    row_pct = 100 * cm / cm.sum(axis=1, keepdims=True)
    ax.imshow(row_pct, cmap='Blues', vmin=0, vmax=100)
    for row in range(2):
        for col in range(2):
            color = 'white' if row_pct[row, col] > 55 else '#17202a'
            ax.text(col, row, f'{cm[row, col]}\n({row_pct[row, col]:.1f}%)',
                    ha='center', va='center', fontsize=12, color=color)
    ax.set(xticks=[0, 1], yticks=[0, 1],
           xticklabels=['V', 'UV'], yticklabels=['V', 'UV'],
           xlabel='Dự đoán', ylabel='Nhãn LAB', title=f'{name} · {int(cm.sum())} khung V/UV')
fig.suptitle('Test V/UV: số khung và % theo nhãn thật')
plt.show()
plt.close(fig)


In [ ]:
# Metric theo file: tách tỷ lệ V/UV và sai số F0 để tránh trộn đơn vị.
# Chỉ minh họa kết quả TEST của cấu hình đã chọn trên training; không dò tham số từ biểu đồ.
plot_groups = (test_summary_df.groupby('Algorithm', sort=False)
               if 'Algorithm' in test_summary_df.columns
               else [('ACF', test_summary_df)])
for algorithm_name, file_results in plot_groups:
    file_results = file_results.reset_index(drop=True)
    names = [name.replace('.wav', '') for name in file_results['file']]
    labels = [f'{name}\nF0={int(n)} / V={int(tp + fn)}'
              for name, n, tp, fn in zip(names, file_results['NumF0'],
                                          file_results['TP'], file_results['FN'])]
    x = np.arange(len(file_results))
    width = 0.35
    f1 = file_results['macro_f1_vu'].to_numpy(dtype=float)
    ba = (file_results['TP'] / (file_results['TP'] + file_results['FN'])
          + file_results['TN'] / (file_results['TN'] + file_results['FP'])).to_numpy(dtype=float) / 2
    mean_error = file_results['abs_error_mean'].to_numpy(dtype=float)
    std_error = file_results['abs_error_std'].to_numpy(dtype=float)
    fig, (top, bottom) = plt.subplots(2, 1, figsize=(9.4, 6.0), sharex=True,
                                      layout='constrained')
    for bars in (top.bar(x - width/2, f1, width, label='Macro F1 V/UV', color='#2166ac'),
                 top.bar(x + width/2, ba, width, label='Balanced accuracy', color='#67a9cf')):
        top.bar_label(bars, fmt='%.2f', padding=2, fontsize=8)
    top.set(ylabel='Tỷ lệ', ylim=(0, 1.18), title=f'{algorithm_name}: phân loại V/UV theo file')
    top.legend(loc='upper left', ncol=2, frameon=False)
    top.grid(axis='y', alpha=0.2)
    top.set_axisbelow(True)
    for bars in (bottom.bar(x - width/2, mean_error, width, label='|F0mean − LAB|', color='#b35806'),
                 bottom.bar(x + width/2, std_error, width, label='|F0std − LAB|', color='#f1a340')):
        bottom.bar_label(bars, fmt='%.1f', padding=2, fontsize=8)
    bottom.set(ylabel='Sai số tuyệt đối (Hz)',
               ylim=(0, max(np.nanmax(mean_error), np.nanmax(std_error)) * 1.35),
               title='Thống kê F0 theo file')
    bottom.set_xticks(x, labels)
    bottom.set_xlabel('F0/V là số đếm, không phải recall V')
    bottom.legend(loc='upper right', ncol=2, frameon=False)
    bottom.grid(axis='y', alpha=0.2)
    bottom.set_axisbelow(True)
    fig.suptitle('TEST: mỗi file có một trọng số như nhau trong macro metric')
    plt.show()
    plt.close(fig)
